In [ ]:
from google.colab import userdata
userdata.get('HuggingFace')

In [ ]:
!git clone -q https://github.com/113119134HAUNG/Dementia.git /content/Dementia || true
!python /content/Dementia/paper/setup_colab.py --all
!touch /content/Dementia/preprocess/__init__.py
!touch /content/Dementia/paper/__init__.py
!touch /content/Dementia/Music_to_text/__init__.py
!touch /content/Dementia/settings/__init__.py
!touch /content/Dementia/tools/__init__.py

In [ ]:
!ls -lh /content
!ls -lh /content/embeddings | head
!ls -lh /content/models | head
!ls -lh /content/Dementia/paper | head
!test -f /content/embeddings/cc.zh.300.vec && echo "[OK] fastText vec ready"

total 1.8G
drwxr-xr-x 8 root root 4.0K Dec 17 02:47 Dementia
drwxr-xr-x 2 root root 4.0K Dec 17 02:49 embeddings
drwxr-xr-x 4 root root 4.0K Dec 17 02:50 hf
drwxr-xr-x 4 root root 4.0K Dec 17 02:53 models
drwxrwxrwx 9 root root 4.0K Dec 16 12:47 NCMMSC2021_AD_Competition-dev
-rw-r--r-- 1 root root 1.8G Dec 16 12:53 NCMMSC2021_AD_Competition-dev.zip
drwxr-xr-x 1 root root 4.0K Dec  9 14:42 sample_data
total 4.3G
-rw-r--r-- 1 root root 4.3G Dec 17 02:49 cc.zh.300.vec
total 8.0K
drwxr-xr-x 3 root root 4.0K Dec 17 02:50 bert-base-chinese
drwxr-xr-x 3 root root 4.0K Dec 17 02:53 google__gemma-2b
total 68K
-rw-r--r-- 1 root root 1.4K Dec 17 02:47 cv_dataset.py
-rw-r--r-- 1 root root 7.1K Dec 17 02:47 cv_eval.py
-rw-r--r-- 1 root root  12K Dec 17 02:47 cv_features.py
-rw-r--r-- 1 root root 2.0K Dec 17 02:47 cv_folds.py
-rw-r--r-- 1 root root 1.1K Dec 17 02:47 cv_utils.py
-rw-r--r-- 1 root root  12K Dec 17 02:47 evaluate_cv.py
-rw-r--r-- 1 root root    1 Dec 17 02:53 __init__.py
-rw-r--r-- 1 r

In [ ]:
%%bash
python - <<'PY'
from pathlib import Path
B=["/content/NCMMSC2021_AD_Competition-dev/dataset",
   "/content/NCMMSC2021_AD_Competition-dev/NCMMSC2021_AD_Competition-dev/dataset"]
S="merge merge_vad raw raw_vad".split(); L="AD HC MCI".split()
E={".wav",".mp3",".flac",".m4a",".ogg"}
cnt=lambda d: sum(f.is_file() and f.suffix.lower() in E for f in d.glob("*"))

for b in B:
  for s in S:
    p=Path(b)/s
    if p.is_dir():
      print("\nROOT:", p)
      for lb in L:
        d=p/lb; print(lb, cnt(d) if d.is_dir() else 0)
PY


ROOT: /content/NCMMSC2021_AD_Competition-dev/dataset/merge
AD 26
HC 44
MCI 53

ROOT: /content/NCMMSC2021_AD_Competition-dev/dataset/merge_vad
AD 26
HC 44
MCI 53

ROOT: /content/NCMMSC2021_AD_Competition-dev/dataset/raw
AD 79
HC 108
MCI 93

ROOT: /content/NCMMSC2021_AD_Competition-dev/dataset/raw_vad
AD 79
HC 108
MCI 93


In [ ]:
%cd /content/Dementia

/content/Dementia


In [ ]:
# ASR（音檔 -> /content/ncmmsc_merged_asr_transcripts.csv）
!python paper/run_pipeline.py --asr --config settings/config_text.yaml

[INFO] TOP LEVEL KEYS: ['common', 'asr', 'predictive', 'text', 'features']

$ /usr/bin/python3 -m Music_to_text.asr_ncmmsc --config settings/config_text.yaml
[INFO] Found 280 audio files under /content/NCMMSC2021_AD_Competition-dev/dataset/raw_vad
[INFO] Loading Whisper model: large-v3 on cuda (float16)
vocabulary.json: 0.00B [00:00, ?B/s]
tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0% 0.00/340 [00:00<?, ?B/s]


preprocessor_config.json: 100% 340/340 [00:00<00:00, 29.7kB/s]
config.json: 2.39kB [00:00, 1.92MB/s]
vocabulary.json: 1.07MB [00:00, 33.0MB/s]
tokenizer.json: 2.48MB [00:00, 63.3MB/s]
model.bin: 100% 3.09G/3.09G [00:39<00:00, 77.7MB/s]
Transcribing: 100% 280/280 [12:14<00:00,  2.62s/file, MCI_M_252308_002.wav]

[INFO] Done. Saved 280 rows to /content/ncmmsc_merged_asr_transcripts.csv (ok=280, error=0)

===== DONE =====
config: settings/config_text.yaml


In [ ]:
# TSV predictive（tsv -> /content/Chinese-predictive_challenge_tsv2_output.jsonl）
!python paper/run_pipeline.py --predictive --config settings/config_text.yaml

[INFO] TOP LEVEL KEYS: ['common', 'asr', 'predictive', 'text', 'features']

$ /usr/bin/python3 -m preprocess.preprocess_predictive --config settings/config_text.yaml
[INFO] Loading meta from: /content/NCMMSC2021_AD_Competition-dev/2_final_list_train.csv
[INFO] Loading eGeMAPS from: /content/NCMMSC2021_AD_Competition-dev/egemaps_final.csv
[INFO] Wrote 323 text records to /content/Chinese-predictive_challenge_tsv2_output.jsonl
[INFO] Merged meta (323) with eGeMAPS (401) -> 323 rows.
[INFO] Saved eGeMAPS feature table to: /content/predictive_egemaps_features.csv

===== DONE =====
config: settings/config_text.yaml


In [ ]:
# Text merge+clean（會用 ASR CSV 產生 ncmmsc_from_asr.jsonl，最後寫 cleaned.jsonl）
!python paper/run_pipeline.py --text --force-asr-jsonl --config settings/config_text.yaml

[INFO] TOP LEVEL KEYS: ['common', 'asr', 'predictive', 'text', 'features']

$ /usr/bin/python3 -m preprocess.preprocess_chinese --config settings/config_text.yaml --force-asr-jsonl
[INFO] Saved NCMMSC JSONL to: /content/chinese_combined/ncmmsc_from_asr.jsonl (n=280)
[INFO] Combining JSONL files:
  - /content/chinese_combined/ncmmsc_from_asr.jsonl
  - /content/Chinese-predictive_challenge_tsv2_output.jsonl
[INFO] Combined JSONL saved to: /content/chinese_combined/Chinese_Combined.jsonl
[INFO] After pre-clean subset: 366 samples remaining.
[INFO] Label distribution after pre-clean subset:
 Diagnosis
HC    219
AD    147
Name: count, dtype: int64
[INFO] Prompt filter (dataset_masked): emptied=100/187, changed=89/187, avg_len_delta=-1.17
[INFO] Quality filter: 366 -> 251 (dropped 115)
[INFO] Quality filter drop reasons (counts, may overlap): short=100, low_lex=100, low_han=115, high_unk=0
[INFO] Label distribution after quality filter:
 Diagnosis
HC    153
AD     98
Name: count, dtype: int6

In [ ]:
import pandas as pd
df = pd.read_json("/content/chinese_combined/cleaned.jsonl", lines=True)
lens = df["Text_interviewer_participant"].fillna("").astype(str).str.len()
print("n=", len(df))
print("empty=", int((lens==0).sum()))
print(df.groupby("Diagnosis")[lens.name].describe())

n= 193
empty= 0
          count unique                               top freq
Diagnosis                                                    
AD           97     97  这个也说吗？ 没有。 嗯这个吗？ 这人不是拿饼干的吗？ 嗯姐姐。    1
HC           96     95    若有單字實在聽不清楚， 請不要亂猜， 若有單字實在聽不清楚，    2


In [ ]:
# CV
!python paper/run_pipeline.py --cv --config settings/config_text.yaml

[INFO] TOP LEVEL KEYS: ['common', 'asr', 'predictive', 'text', 'features']

$ /usr/bin/python3 -m paper.evaluate_cv --config settings/config_text.yaml
[INFO] Saved fold indices to: /content/chinese_combined/folds_indices.json (n_folds=5)

Fold 1 (tfidf train-only):
              precision    recall  f1-score   support

          HC       0.60      0.79      0.68        19
          AD       0.71      0.50      0.59        20

    accuracy                           0.64        39
   macro avg       0.66      0.64      0.64        39
weighted avg       0.66      0.64      0.63        39

[Confusion Matrix] Fold 1 (tfidf train-only) labels=['HC', 'AD']: [[15, 4], [10, 10]]

Fold 2 (tfidf train-only):
              precision    recall  f1-score   support

          HC       0.64      0.74      0.68        19
          AD       0.71      0.60      0.65        20

    accuracy                           0.67        39
   macro avg       0.67      0.67      0.67        39
weighted avg       0.